In [44]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter Groq API Key: ")


Enter Groq API Key: ··········


In [45]:
!pip install -q langchain-groq langchain-core requests

In [46]:
from langchain_groq import ChatGroq

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

import requests

In [47]:
!pip install -q langchain-huggingface langchain-core requests

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

import requests

In [48]:
#tool create
@tool
def multiply(a:int,b:int)->int:
  """ given two numbers A and B this tool return thier product"""
  return a*b

In [49]:
print(multiply.invoke({'a':3,'b':4}))

12


In [50]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [51]:
model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [52]:
llm_with_tools=model.bind_tools([multiply])

In [53]:
#tool
from langchain_core.tools import InjectedToolArg
from typing import Annotated
@tool
def get_conversion_factor(base_currency:str,target_currency:str)->float:
  """
  this function fetches the currency conversion factor between the base currency and the target currency
  """
  url=f'https://v6.exchangerate-api.com//pair/{base_currency}/{target_currency}'
  response=requests.get(url)
  return response.json()

@tool
def convert(base_currency:int,conversion_rate:Annotated[float,InjectedToolArg])->float:
  """
  given a currency conversion rate this function converts the base currency to the target currency
  """
  return base_currency*conversion_rate


In [54]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1793232001,
 'time_last_update_utc': 'Thu, 29 Oct 2026 00:00:01 +0000',
 'time_next_update_unix': 1793318401,
 'time_next_update_utc': 'Fri, 30 Oct 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.2911}

In [55]:
convert.invoke({'base_currency': 10, 'conversion_rate': 95.153})

951.5300000000001

In [56]:
#tool binding
llm_with_tools=model.bind_tools([get_conversion_factor,convert])

In [57]:
message=[HumanMessage('what is the conversion factor between USD to INR based on that you can convert 10USD to inr')]

In [58]:
message

[HumanMessage(content='what is the conversion factor between USD to INR based on that you can convert 10USD to inr', additional_kwargs={}, response_metadata={})]

In [59]:
ai_message=llm_with_tools.invoke(message)

In [60]:
message.append(ai_message)

In [61]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'fc_34b6385a-0d14-4bac-a978-47ce6b436f10',
  'type': 'tool_call'}]

In [62]:
import json
for tool_call in ai_message.tool_calls:
  #first ham tool 1 ko run krwange and uski o/p ko second tool call ko denge
  if(tool_call['name']=='get_conversion_factor'):
    tool_message1=get_conversion_factor.invoke(tool_call)
  #fetch karenge conversion rate agle tool call ke liye then append kr denge
  conversion_rate=json.loads(tool_message1.content)['conversion_rate']
  #append kr denge msg me
  message.append(tool_message1)
  #ab second tool ke liye
  if(tool_call['name']=='convert'):
    tool_call['args']['conversion_rate']=conversion_rate
    tool_message2=convert.invoke(tool_call)
    message.append(tool_message2)


In [63]:
message

[HumanMessage(content='what is the conversion factor between USD to INR based on that you can convert 10USD to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "what is the conversion factor between USD to INR based on that you can convert 10USD to inr". They want the conversion factor. We need to fetch conversion factor. Use get_conversion_factor function. Provide base_currency: "USD", target_currency: "INR".', 'tool_calls': [{'id': 'fc_34b6385a-0d14-4bac-a978-47ce6b436f10', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 189, 'total_tokens': 281, 'completion_time': 0.108769758, 'completion_tokens_details': {'reasoning_tokens': 61}, 'prompt_time': 0.010705783, 'prompt_tokens_details': None, 'queue_time': 0.168824596, 'total_time': 0.119475541}, 

In [64]:
llm_with_tools.invoke(message).content

'The current conversion factor from USD to INR is **95.2911**.  \nSo, converting\u202f$10\u202fUSD\u202fto\u202fINR would give:\n\n\\[\n10 \\text{ USD} \\times 95.2911 \\;(\\text{INR/USD}) = 952.911 \\text{ INR}\n\\]'